In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, ReLU, MaxPool1D, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load Dataset Hasil GA
df = pd.read_csv('../dataset/processed/ga_optimized_features_dataset.csv')
X = df.drop(columns=['label'])
y = df['label']

# 2. Split Data (Konsisten 80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Wajib Scaling untuk Deep Learning
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Reshape dari 2D menjadi 3D (samples, 38, 1)
X_train_cnn = np.expand_dims(X_train_scaled, axis=-1)
X_test_cnn = np.expand_dims(X_test_scaled, axis=-1)

# 5. Menangani Mild Imbalance (Imbalance Ratio 1.92)
class_weights = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(y_train), 
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

# 6. Arsitektur Model 1D-CNN
model = Sequential([
    # Blok Konvolusi 1
    Conv1D(filters=32, kernel_size=3, padding='same', input_shape=(X_train_cnn.shape[1], 1)),
    BatchNormalization(),
    ReLU(),
    MaxPool1D(pool_size=2),
    Dropout(0.2),
    
    # Blok Konvolusi 2
    Conv1D(filters=64, kernel_size=3, padding='same'),
    BatchNormalization(),
    ReLU(),
    MaxPool1D(pool_size=2),
    Dropout(0.3),
    
    # Fully Connected Layer
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid') # Output biner
])

# 7. Compile Model dengan Metrik Evaluasi Lengkap
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
)

# 8. Early Stopping untuk Mencegah Overfitting
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True
)

# 9. Training Model
history = model.fit(
    X_train_cnn, y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2, # Internal validation split
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

I0000 00:00:1783778169.034646   14424 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783778169.435115   14424 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783778170.993900   14424 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/dionazani/budi-luhur-2025/semester_tiga/kecerdasan_komputasional/my-midterm_exam-midmalcurl-202607071300/

Epoch 1/50
6386/6386 ━━━━━━━━━━━━━━━━━━━━ 37s 6ms/step - accuracy: 0.8892 - auc: 0.9398 - loss: 0.2988 - recall: 0.8006 - val_accuracy: 0.9166 - val_auc: 0.9609 - val_loss: 0.2253 - val_recall: 0.8434
Epoch 2/50
6386/6386 ━━━━━━━━━━━━━━━━━━━━ 35s 5ms/step - accuracy: 0.9072 - auc: 0.9549 - loss: 0.2594 - recall: 0.8269 - val_accuracy: 0.9153 - val_auc: 0.9631 - val_loss: 0.2252 - val_recall: 0.8591
Epoch 3/50
6386/6386 ━━━━━━━━━━━━━━━━━━━━ 38s 6ms/step - accuracy: 0.9104 - auc: 0.9577 - loss: 0.2508 - recall: 0.8353 - val_accuracy: 0.9115 - val_auc: 0.9651 - val_loss: 0.2277 - val_recall: 0.8727
Epoch 4/50
6386/6386 ━━━━━━━━━━━━━━━━━━━━ 35s 5ms/step - accuracy: 0.9123 - auc: 0.9594 - loss: 0.2454 - recall: 0.8395 - val_accuracy: 0.9181 - val_auc: 0.9658 - val_loss: 0.2091 - val_recall: 0.8640
Epoch 5/50
6386/6386 ━━━━━━━━━━━━━━━━━━━━ 36s 6ms/step - accuracy: 0.9129 - auc: 0.9603 - loss: 0.2425 - recall: 0.8424 - val_accuracy: 0.9217 - val_auc: 0.9664 - val_loss: 0.2066 - val_recall: 0.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# 1. Prediksi Probabilitas pada Test Set
y_pred_prob = model.predict(X_test_cnn)

# 2. Konversi Probabilitas menjadi Label Biner (Threshold 0.5)
y_pred = (y_pred_prob > 0.5).astype(int)

# 3. Cetak Classification Report Resmi
print("=================== CLASSIFICATION REPORT 1D-CNN ===================")
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malicious']))

# 4. Hitung Akhir Nilai ROC-AUC
test_auc = roc_auc_score(y_test, y_pred_prob)
print(f"Test Set ROC-AUC Score: {test_auc:.4f}")
print("===================================================================\n")

# 5. Visualisasi Confusion Matrix Mandiri
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Benign', 'Malicious'], 
            yticklabels=['Benign', 'Malicious'], ax=ax)
ax.set_title('Confusion Matrix - 1D-CNN Model', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.show()